# `dman` — Practical Walkthrough

Self-contained, end-to-end tutorial for the `dman` dataset manager.  
Everything runs locally — no large downloads. We generate a tiny synthetic dataset inline.

### What you'll learn
1. Set up an isolated demo catalog
2. Create a mini YOLO dataset with PIL
3. Import, list, and inspect via the CLI
4. Load and iterate with the Python SDK
5. Access annotations
6. Visualise a sample with matplotlib
7. Update the dataset (add samples + annotations)
8. Export to COCO format
9. Zero-copy Arrow access
10. HuggingFace / PyTorch interop
11. Clean up

### Prerequisites
- Python 3.9+
- `dman` installed (`pip install .` from repo root — requires Rust toolchain)
- `Pillow`, `matplotlib` — `pip install pillow matplotlib`

## 1 — Setup

We point `DMAN_HOME` to `/tmp/dman_demo` so this notebook never touches your real catalog.  
**Must be set before importing `dman`.**

In [ ]:
import os, sys

os.environ["DMAN_HOME"] = "/tmp/dman_demo"

print(f"DMAN_HOME = {os.environ['DMAN_HOME']}")
print(f"Python    = {sys.version}")

In [ ]:
! dman init

## 2 — Create a Synthetic YOLO Dataset

5 images (32×32 PNGs), 5 label files, and a `data.yaml`.  
Two classes: `cat` (0) and `dog` (1).

In [ ]:
import pathlib
from PIL import Image

YOLO_ROOT = pathlib.Path("/tmp/dman_yolo_demo")
IMG_DIR   = YOLO_ROOT / "images" / "train"
LBL_DIR   = YOLO_ROOT / "labels" / "train"

IMG_DIR.mkdir(parents=True, exist_ok=True)
LBL_DIR.mkdir(parents=True, exist_ok=True)

COLORS  = [(220, 80, 80), (80, 180, 80), (80, 120, 220), (220, 200, 60), (160, 80, 200)]
CLASSES = [0, 1, 0, 1, 0]
BOXES   = [
    (0.50, 0.50, 0.40, 0.40),
    (0.30, 0.30, 0.30, 0.30),
    (0.60, 0.40, 0.35, 0.45),
    (0.45, 0.55, 0.50, 0.30),
    (0.55, 0.45, 0.25, 0.35),
]

for i, (color, cls, box) in enumerate(zip(COLORS, CLASSES, BOXES)):
    Image.new("RGB", (32, 32), color=color).save(IMG_DIR / f"img{i}.png")
    cx, cy, w, h = box
    (LBL_DIR / f"img{i}.txt").write_text(f"{cls} {cx} {cy} {w} {h}\n")

(YOLO_ROOT / "data.yaml").write_text(
    "path: .\ntrain: images/train\nnc: 2\nnames:\n  0: cat\n  1: dog\n"
)

print(f"Dataset written to {YOLO_ROOT}")
for p in sorted(YOLO_ROOT.rglob("*")):
    print(f"  {p.relative_to(YOLO_ROOT)}")

## 3 — Import via CLI

In [ ]:
! dman import /tmp/dman_yolo_demo --format yolo --name my_demo

## 4 — Browse the Catalog

In [ ]:
! dman list

In [ ]:
! dman inspect my_demo

## 5 — Load with the Python SDK

`dman.load_dataset()` returns a `DmanDataset` — an eagerly-loaded, read-only snapshot.  
No data is copied; asset paths point to the original files on disk.

The SDK returns plain dicts — easy to inspect, filter, and feed into any pipeline.

In [ ]:
import dman

ds = dman.load_dataset("my_demo")

print(f"name           : {ds.name}")
print(f"dataset_id     : {ds.dataset_id}")
print(f"len(ds)        : {len(ds)}")
print(f"sample_count   : {ds.sample_count()}")
print(f"asset_count    : {ds.asset_count()}")
print(f"annotation_count: {ds.annotation_count()}")
print(f"category_count : {ds.category_count()}")

## 6 — Iterate Samples

`ds.samples()` returns a list of dicts, each with `id`, `name`, `metadata`, and `assets`.

In [ ]:
for s in ds.samples():
    img_paths = [a["file_path"] for a in s["assets"] if a["asset_type"] == "image"]
    print(f"  Sample '{s['name']}'  id={s['id']}  images={img_paths}")

In [ ]:
# Indexing works too: ds[idx] returns a dict with assets + annotations
first = ds[0]
print("Keys       :", list(first.keys()))
print("Assets     :", [{"type": a["asset_type"], "file": a["file_name"]} for a in first["assets"]])
print("Annotations:", len(first["annotations"]))

In [ ]:
# Lookup by name
sample = ds.get_sample("img0.png")
if sample:
    print(f"Found: id={sample['id']}, name={sample['name']}, assets={len(sample['assets'])}")
else:
    print("Sample not found")

## 7 — Access Annotations

`ds.annotations(sample_name)` returns annotation dicts.  
The `bbox` field is a JSON string — parse it if you need the coordinates.

In [ ]:
import json

for s in ds.samples():
    for ann in ds.annotations(s["name"]):
        bbox_raw = ann.get("bbox")
        if bbox_raw:
            bbox = json.loads(bbox_raw) if isinstance(bbox_raw, str) else bbox_raw
            cat_id = ann.get("category_id", "?")
            print(f"  [{s['name']}]  category_id={cat_id}  bbox={bbox}")

## 8 — Visualise a Sample

Open the first image, overlay the bounding box with matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Pick the first sample
target = ds.samples()[0]
img_asset = next((a for a in target["assets"] if a["asset_type"] == "image"), None)
if img_asset is None:
    raise RuntimeError("No image asset found")

anns = ds.annotations(target["name"])

# Load and scale up for visibility
img = Image.open(img_asset["file_path"]).resize((256, 256), Image.NEAREST)
img_w, img_h = img.size

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(np.array(img))
ax.set_title(f"Sample: {target['name']}")
ax.axis("off")

for ann in anns:
    bbox_raw = ann.get("bbox")
    if not bbox_raw:
        continue
    bbox = json.loads(bbox_raw) if isinstance(bbox_raw, str) else bbox_raw

    # bbox is stored as {x, y, width, height} in pixel coords
    scale_x = img_w / 32
    scale_y = img_h / 32
    x = bbox["x"] * scale_x
    y = bbox["y"] * scale_y
    w = bbox["width"] * scale_x
    h = bbox["height"] * scale_y

    rect = mpatches.Rectangle(
        (x, y), w, h,
        linewidth=2, edgecolor="lime", facecolor="none",
    )
    ax.add_patch(rect)
    ax.text(x, y - 4, f"cat_id={ann.get('category_id', '?')}",
            color="lime", fontsize=9,
            bbox=dict(facecolor="black", alpha=0.5, pad=1))

out_path = "/tmp/dman_demo_vis.png"
plt.savefig(out_path, bbox_inches="tight", dpi=96)
plt.show()
print(f"Saved → {out_path}")

## 9 — Update the Dataset

`dman.update_dataset()` returns an updater for transactional mutations.  
Add samples, assets, annotations — then commit with `.apply()`.

In [ ]:
# Create one extra image
extra_img = "/tmp/dman_extra_img.png"
Image.new("RGB", (32, 32), color=(255, 160, 0)).save(extra_img)

updater = dman.update_dataset("my_demo")
updater.add_sample("extra_sample", metadata={"source": "manual"})
updater.add_asset(
    sample_name="extra_sample",
    asset_type="image",
    file_path=extra_img,
    width=32,
    height=32,
)
updater.apply()
print("Update applied.")

# Reload to verify
ds = dman.load_dataset("my_demo")
print(f"Sample count: {ds.sample_count()}  (was 5, now 6)")

In [ ]:
# Add an annotation to the new sample (requires sample_id)
extra = ds.get_sample("extra_sample")
if extra:
    updater2 = dman.update_dataset("my_demo")
    updater2.add_annotation(
        sample_id=extra["id"],
        category="cat",
        bbox=[4.0, 4.0, 24.0, 24.0],
    )
    updater2.apply()
    print("Annotation added.")

    # Verify
    ds = dman.load_dataset("my_demo")
    anns = ds.annotations("extra_sample")
    print(f"Annotations on extra_sample: {len(anns)}")
    for ann in anns:
        print(f"  bbox={ann.get('bbox')}  category_id={ann.get('category_id')}")

## 10 — Export to COCO Format

In [ ]:
import shutil

COCO_OUT = "/tmp/dman_coco_output"
shutil.rmtree(COCO_OUT, ignore_errors=True)

! dman export my_demo /tmp/dman_coco_output --format coco

In [ ]:
coco_files = sorted(pathlib.Path(COCO_OUT).rglob("*.json"))
print(f"Exported: {[str(f) for f in coco_files]}")

if coco_files:
    coco = json.loads(coco_files[0].read_text())

    print(f"\nCategories ({len(coco.get('categories', []))})")
    for cat in coco.get("categories", []):
        print(f"  {cat}")

    print(f"\nImages (first 3 of {len(coco.get('images', []))})")
    for img in coco.get("images", [])[:3]:
        print(f"  {img}")

    print(f"\nAnnotations (first 3 of {len(coco.get('annotations', []))})")
    for ann in coco.get("annotations", [])[:3]:
        print(f"  {ann}")

## 11 — Arrow Zero-Copy Access

For structured, columnar access without dict overhead, use the Arrow methods.  
These return `pyarrow.RecordBatch` objects via zero-copy FFI — no serialisation cost.

Requires `pyarrow` (`pip install pyarrow`).

In [ ]:
try:
    import pyarrow  # noqa: F401

    tables = ds.to_arrow()
    for name, batch in tables.items():
        print(f"{name:15s}  rows={batch.num_rows}  cols={batch.num_columns}")

    print("\n--- samples_arrow schema ---")
    print(ds.samples_arrow().schema)

    print("\n--- assets_arrow (as pandas) ---")
    print(ds.assets_arrow().to_pandas().head())

except ImportError:
    print("[skip] pyarrow not installed — pip install pyarrow")

## 12 — HuggingFace & PyTorch Interop

Optional — wrapped in `try/except` so the notebook still runs without these packages.

In [ ]:
try:
    hf_ds = ds.to_hf_dataset()
    print("HuggingFace dataset:", hf_ds)
    print("First row:          ", hf_ds[0])
except ImportError:
    print("[skip] `datasets` not installed — pip install datasets")
except Exception as e:
    print(f"[skip] HuggingFace conversion: {e}")

In [ ]:
try:
    torch_ds = ds.to_torch_dataset()
    print("PyTorch dataset length:", len(torch_ds))
    print("First item:           ", torch_ds[0])
except ImportError:
    print("[skip] `torch` not installed — pip install torch")
except Exception as e:
    print(f"[skip] PyTorch conversion: {e}")

## 13 — Cleanup

In [ ]:
! dman remove my_demo -y

In [ ]:
! dman list

In [ ]:
for path in ["/tmp/dman_yolo_demo", "/tmp/dman_coco_output", "/tmp/dman_demo"]:
    shutil.rmtree(path, ignore_errors=True)
    print(f"Removed {path}")

for f in ["/tmp/dman_extra_img.png", "/tmp/dman_demo_vis.png"]:
    p = pathlib.Path(f)
    if p.exists():
        p.unlink()
        print(f"Removed {f}")

print("\nDone — catalog and temp files cleaned up.")

---

## Summary

| Step | What we did |
|------|-------------|
| 1 | Set `DMAN_HOME`, ran `! dman init` |
| 2 | Generated 5 synthetic images + YOLO labels |
| 3 | Imported via `! dman import` |
| 4 | Browsed with `! dman list` / `! dman inspect` |
| 5 | Loaded with `dman.load_dataset()` |
| 6 | Iterated samples and assets as dicts |
| 7 | Accessed annotations via `ds.annotations()` |
| 8 | Visualised bboxes with matplotlib |
| 9 | Updated the dataset via `dman.update_dataset()` |
| 10 | Exported to COCO JSON |
| 11 | Zero-copy Arrow access (`ds.to_arrow()`) |
| 12 | Optional HuggingFace / PyTorch interop |
| 13 | Cleaned up catalog and temp files |

### Next steps
- Import your own dataset: `dman import <path> --format yolo --name mydata`
- Try `--format coco` and `--format huggingface` imports
- Define a custom schema for metadata validation
- Write a Python plugin (`$DMAN_HOME/plugins/`) for custom formats
- Serve the catalog: `dman server start`